In [ ]:
import pandas as pd 
import numpy as np
# import matplotlib.pyplot as plt 
# import seaborn as sns
import torch
from functools import partial
# from transformers import AutoTokenizer, AutoModel, AutoModelForCausalLM

import sys, os
project_root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
if project_root not in sys.path:
    sys.path.append(project_root)

import asyncio
from lightrag.utils import setup_logger, EmbeddingFunc
from lightrag.llm.hf import hf_embed
from lightrag import LightRAG, QueryParam
from lightrag.llm.openai import gpt_4o_mini_complete, gpt_4o_complete, openai_embed
from lightrag.utils import setup_logger

In [2]:
setup_logger("lightrag", level="INFO")

WORKING_DIR = "./rag_storage"
if not os.path.exists(WORKING_DIR):
    os.mkdir(WORKING_DIR)

In [ ]:
# embed_tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")
# embed_model = AutoModel.from_pretrained("sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
# MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# llm_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_NAME,
#     torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
#     device_map="auto"
# )

In [ ]:
# async def hf_model_complete(prompt: str, **kwargs):
#     device = model.device
    
#     inputs = llm_tokenizer(prompt, return_tensors="pt")
    
#     # Remove token_type_ids if they exist (TinyLlama/Llama models don't use them)
#     inputs.pop("token_type_ids", None)
    
#     inputs = {k: v.to(device) for k, v in inputs.items()}
    
#     outputs = model.generate(
#         **inputs,
#         max_new_tokens=256,
#         temperature=0.7,
#         do_sample=True,
#         pad_token_id=llm_tokenizer.eos_token_id # Ensure padding is set
#     )

#     # Decode and remove the prompt from the output
#     full_text = llm_tokenizer.decode(outputs[0], skip_special_tokens=True)
#     return full_text[len(prompt):].strip() if full_text.startswith(prompt) else full_text

In [ ]:
## This version uses system prompt to guide the model, actually I don't know if LightRAG itself has prebuilt system prompt, but if not then write it

async def hf_model_complete(prompt: str, system_prompt=None, history_messages=[], **kwargs):
    device = model.device
    messages = []
    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})
    messages.extend(history_messages)
    messages.append({"role": "user", "content": prompt})
    
    # Use chat template to wrap the complex LightRAG instructions
    tokenized_chat = llm_tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt")
    inputs = {"input_ids": tokenized_chat.to(device)}

    outputs = model.generate(
        **inputs,
        max_new_tokens=512,
        temperature=0.1,
        do_sample=True,
        pad_token_id=llm_tokenizer.eos_token_id
    )

    decoded = llm_tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return decoded.strip()

In [ ]:
# Using light-weight Huggingface model

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=hf_model_complete,
    llm_model_name=MODEL_NAME, # TinyLlama
    embedding_func=EmbeddingFunc(
        embedding_dim=384,
        max_token_size=400,
        model_name="sentence-transformers/all-MiniLM-L6-v2",
        func=partial(
            hf_embed.func,
            embed_model=embed_model, 
            tokenizer=embed_tokenizer    
        )
    )
)

INFO: [] Created new empty graph file: ./rag_storage\graph_chunk_entity_relation.graphml
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './rag_storage\\vdb_entities.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './rag_storage\\vdb_relationships.json'} 0 data
INFO:nano-vectordb:Init {'embedding_dim': 384, 'metric': 'cosine', 'storage_file': './rag_storage\\vdb_chunks.json'} 0 data
INFO: [] Process 11228 KV load full_docs with 0 records
INFO: [] Process 11228 KV load text_chunks with 0 records
INFO: [] Process 11228 KV load full_entities with 0 records
INFO: [] Process 11228 KV load full_relations with 0 records
INFO: [] Process 11228 KV load entity_chunks with 0 records
INFO: [] Process 11228 KV load relation_chunks with 0 records
INFO: [] Process 11228 KV load llm_response_cache with 0 records
INFO: [] Process 11228 doc status load doc_status with 2 records


In [ ]:
# Using vLLM

rag = LightRAG(
    working_dir=WORKING_DIR,
    llm_model_func=openai_complete,
    llm_model_name="BioMistral/BioMistral-7B-AWQ-QGS128-W4-GEMM",
    llm_model_max_async=4,
    llm_model_kwargs={
        "base_url": "http://127.0.0.1:8080/v1",
        "api_key": "none" 
    },
    
    embedding_func=EmbeddingFunc(
        embedding_dim=768, # Change based on dim of embedding model
        max_token_size=8192,
        func=partial(
            openai_embed,
            base_url="http://127.0.0.1:8081/v1",
            api_key="none",
            model="nomic-ai/nomic-embed-text-v1.5"
        )
    )
)

In [ ]:
await rag.initialize_storages()

In [ ]:
await rag.ainsert("""

What is anatomy?

Anatomy includes those structures that can be seen grossly (without the aid of magnification) and microscopically (with the aid of magnification). Typically, when used by itself, the term anatomy tends to mean gross or macroscopic anatomy—that is, the study of structures that can be seen without using a microscopic. Microscopic anatomy, also called histology, is the study of cells and tissues using a microscope.

Anatomy forms the basis for the practice of medicine. Anatomy leads the physician toward an understanding of a patient’s disease, whether he or she is carrying out a physical examination or using the most advanced imaging techniques. Anatomy is also important for dentists, chiropractors, physical therapists, and all others involved in any aspect of patient treatment that begins with an analysis of clinical signs. The ability to interpret a clinical observation correctly is therefore the endpoint of a sound anatomical understanding.

Observation and visualization are the primary techniques a student should use to learn anatomy. Anatomy is much more than just memorization of lists of names. Although the language of anatomy is important, the network of information needed to visualize the position of physical structures in a patient goes far beyond simple memorization. Knowing the names of the various branches of the external carotid artery is not the same as being able to visualize the course of the lingual artery from its origin in the neck to its termination in the tongue. Similarly, understanding the organization of the soft palate, how it is related to the oral and nasal cavities, and how it moves during swallowing is very different from being able to recite the names of its individual muscles and nerves. An understanding of anatomy requires an understanding of the context in which the terminology can be remembered.

How can gross anatomy be studied?

The term anatomy is derived from the Greek word temnein, meaning “to cut.” Clearly, therefore, the study of anatomy is linked, at its root, to dissection, although dissection of cadavers by students is now augmented, or even in some cases replaced, by viewing prosected (previously dissected) material and plastic models, or using computer teaching modules and other learning aids.

Anatomy can be studied following either a regional or a systemic approach.

With a regional approach, each region of the body is studied separately and all aspects of that region are studied at the same time. For example, if the thorax is to be studied, all of its structures are examined.

This includes the vasculature, the nerves, the bones, the muscles, and all other structures and organs located in the region of the body defined as the thorax. After studying this region, the other regions of the body (i.e., the abdomen, pelvis, lower limb, upper limb, back, head, and neck) are studied in a similar fashion.

In contrast, in a systemic approach, each system of the body is studied and followed throughout the entire body. For example, a study of the cardiovascular system looks at the heart and all of the blood vessels in the body. When this is completed, the nervous system (brain, spinal cord, and all the nerves) might be examined in detail. This approach continues for the whole body until every system, including the nervous, skeletal, muscular, gastrointestinal, respiratory, lymphatic, and reproductive systems, has been studied.

Each of these approaches has benefits and deficiencies. The regional approach works very well if the anatomy course involves cadaver dissection but falls short when it comes to understanding the continuity of an entire system throughout the body. Similarly, the systemic approach fosters an understanding of an entire system throughout the body, but it is very difficult to coordinate this directly with a cadaver dissection or to acquire sufficient detail.

""")

INFO: Preserving 2 failed document entries for manual review
INFO: Processing 1 document(s)
INFO: Extracting stage 1/1: unknown_source
INFO: Processing d-id: doc-dc5546c383ca2a5888cbcaf2e5fc53c1
INFO: Embedding func: 8 new workers initialized (Timeouts: Func: 30s, Worker: 60s, Health Check: 75s)
INFO: LLM func: 4 new workers initialized (Timeouts: Func: 180s, Worker: 360s, Health Check: 375s)
Token indices sequence length is longer than the specified maximum sequence length for this model (4378 > 2048). Running this sequence through the model will result in indexing errors
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
This is a friendly reminder - the current text generation call will exceed the model's predefined maximum length (2048). Depending on the model, you may observe exceptions, performance degrad

'insert_20260311_091454_e4e64d1a'

#### Test Retrieve

In [12]:
query_1 = '''
Compare and contrast the regional and systemic approaches to studying anatomy. Under what specific circumstances would one approach be preferred over the other?
'''

query_2 = '''
How does the etymological origin of the word 'anatomy' relate to the primary techniques used to study it, and what modern alternatives have been introduced to supplement traditional methods?
'''

query_3 = '''
What is anatomy?
'''

In [15]:
mode = "global"

response = await rag.aquery(
        query_3,
        param=QueryParam(mode=mode)
    )

INFO:  == LLM cache == saving: global:keywords:5f2b054463e5df9546145df925e5018a
INFO: Query edges: Anatomy, Body parts, Muscles, Bones, Skin (top_k:40, cosine:0.2)
INFO: Raw search results: 0 entities, 0 relations, 0 vector chunks
INFO: [kg_query] No query context could be built; returning no-result.
